In [2]:
import pandas as pd
import plotly.express as px
from movie_analytics.database import get_engine

In [3]:
engine = get_engine()
movies_df = pd.read_sql("SELECT * FROM movies_full_dataset",engine)
movies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6434 entries, 0 to 6433
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   imdb_id     6434 non-null   str    
 1   title       6434 non-null   str    
 2   year        6434 non-null   int64  
 3   genres      6432 non-null   str    
 4   avg_rating  6434 non-null   float64
 5   votes       6434 non-null   int64  
 6   popularity  6434 non-null   float64
 7   revenue     6434 non-null   int64  
 8   budget      6434 non-null   int64  
dtypes: float64(2), int64(4), str(3)
memory usage: 713.7 KB


In [4]:
df_cleaned = movies_df.dropna(subset=["genres"]).copy()
df_cleaned["genres_list"] = df_cleaned["genres"].str.split(',')
df_cleaned["genres_weight"] = 1/df_cleaned["genres_list"].apply(len)
df_exploded = df_cleaned.explode("genres_list")
df = df_exploded.rename(columns={"genres_list":"genre","genres_weight":"genre_weight"})
df


,imdb_id,title,year,genres,avg_rating,votes,popularity,revenue,budget,genre,genre_weight
0,tt0468569,The Dark Knight,2008,"Crime,Thriller",9.1,3174396,130.6430,1004558444,185000000,Crime,0.500000
0,tt0468569,The Dark Knight,2008,"Crime,Thriller",9.1,3174396,130.6430,1004558444,185000000,Thriller,0.500000
1,tt1375666,Inception,2010,"Adventure,Sci-Fi,Thriller",8.8,2822508,83.9520,825532764,160000000,Adventure,0.333333
1,tt1375666,Inception,2010,"Adventure,Sci-Fi,Thriller",8.8,2822508,83.9520,825532764,160000000,Sci-Fi,0.333333
1,tt1375666,Inception,2010,"Adventure,Sci-Fi,Thriller",8.8,2822508,83.9520,825532764,160000000,Thriller,0.333333
...,...,...,...,...,...,...,...,...,...,...,...
6429,tt12833012,Venalmaram,2009,Drama,3.0,7,0.6000,1000,1000,Drama,1.000000
6430,tt7220688,Sabse Bada Mujrim,2013,Drama,5.5,7,0.0000,550000,241000,Drama,1.000000
6431,tt9536384,Mai Mai Ti's 2008,2008,Drama,6.8,7,0.6000,20707,414147,Drama,1.000000
6432,tt36955396,Mamá quiero ser futbolista profesional,2022,Documentary,9.7,7,0.0357,3000,15000,Documentary,1.000000


In [5]:
genres_list = ["Action", "Adventure", "Animation", "Comedy", "Drama", "Fantasy", "Horror", "Mystery", "Romance", "Sci-Fi", "Thriller"]

colors = ["#FF5252", "#38EF7D", "#FFD700", "#00E5FF", "#E040FB", "#FF9100", "#5C5CFF", "#FF4081", "#00F5D4", "#B388FF", "#CCFF00"]

genre_color_map = dict(zip(genres_list,colors))



In [9]:
df_ratings = df.copy()
df_ratings = df_ratings[df_ratings["genre"].isin(genres_list)]
df_ratings = df_ratings.groupby("genre", as_index=False).agg(mean_rating=("avg_rating","mean"))[["genre","mean_rating"]]

fig=px.bar(df_ratings,
        x="genre", 
        y="mean_rating",
        color="genre",
        color_discrete_map=genre_color_map
    )

fig.show()




In [ ]:
df_ratings = df.copy()
df_ratings = df_ratings[df_ratings["genre"].isin(genres_list)]

fig=px.box(df_ratings,
        x="genre", 
        y="avg_rating",
        color="genre",
        color_discrete_map=genre_color_map,
        category_orders={"genre": genres_list}
    )

fig.show()

In [ ]:
df_revenue_total = df.copy()
df_revenue_total = df_revenue_total[df_revenue_total["genre"].isin(genres_list)]
df_revenue_total["weighted_revenue"] = df_revenue_total["revenue"]*df_revenue_total["genre_weight"]
df_revenue_total = df_revenue_total.groupby("genre",as_index=False).agg(sum_weighted_revenue=("weighted_revenue","sum"))[["genre","sum_weighted_revenue"]]
df_revenue_total["weighted_revenue_share"] = (df_revenue_total["sum_weighted_revenue"])*100/(df_revenue_total["sum_weighted_revenue"].sum())

fig = px.bar(
    df_revenue_total,
    x="genre",
    y="weighted_revenue_share",
    color="genre",
    color_discrete_map=genre_color_map
)

fig.show()

In [ ]:
df_revenue_trend = df.copy()
df_revenue_trend = df_revenue_trend[df_revenue_trend["genre"].isin(genres_list)]
df_revenue_trend["weighted_revenue"] = df_revenue_trend["revenue"]*df_revenue_trend["genre_weight"]
df_revenue_trend = df_revenue_trend.groupby(["year","genre"],as_index=False).agg(sum_weighted_revenue=("weighted_revenue","sum"))[["year","genre","sum_weighted_revenue"]]
df_revenue_trend["weighted_revenue_share"] = (df_revenue_trend["sum_weighted_revenue"])*100/(df_revenue_trend.groupby("year")["sum_weighted_revenue"].transform("sum"))

fig = px.line(
    df_revenue_trend,
    x="year",
    y="weighted_revenue_share",
    color="genre",
    color_discrete_map=genre_color_map
)

fig.update_layout(
    legend_itemclick="toggleothers", 
    legend_itemdoubleclick="toggle"      
)

fig.show()